# 20 — Figure Selection (label-driven pre-pre-processing)

Thin notebook: it only **imports**, **calls** `src/figure_selection.py`, and **displays**.
It decides **which figures** enter the embedding pipeline using only the human labels. It does not open any image.

**Input:** Stage 04's join in `paths.labelled_root` (`joined/master_labels.xlsx`, `joined/master_figures.xlsx`).
**Rules:** the `selection:` block of `config.yaml`. The reason for each rule is in `eVTOL-Visual-Evaluation/docs/embedding_evaluation/SELECTION_DECISIONS.md`.
**Output:** `<paths.pipeline_root>/selection/`: `candidates.csv`, `funnel.csv`, `aircraft.parquet`, `coverage.csv`, `sets/<name>.csv`, `sets_union.csv` and `selection_summary.json`.

The steps:
1. Build the candidate table: every figure on file, with its own labels and its aircraft's labels.
2. Apply the fixed gates, the same for every set, which gives the funnel.
3. Build one figure set per strategy.
4. Check coverage, and split the aircraft into a select half and a report half.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'config.yaml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from src.config_loader import load_config
from src import figure_selection as fs

cfg = load_config()
master, figures = fs.load_master(cfg)
print(len(master), 'aircraft rows |', len(figures), 'figure rows')

1750 aircraft rows | 5855 figure rows


## 1. Candidate table

In [2]:
cand = fs.build_candidates(master, figures)
cand.head()

,batch,patent_id,block,fig_key,arch,image_file,approved_copy_path,status,is_main,per,...,dup_type,same_aircraft_as,labels_inherited_from,t1_humanUncertain,edgeTags,arch_gt,arch_gt_visible,aircraft_uid,patent_approved,fig_order
0,Batch_02,AT503689A1,Image: AT503689A1_fig_01_crop_0_F2.png,2,1,AT503689A1_fig_01_crop_0_F2.png,/mnt/storage_11tb/Drive_files_to_syncronize/3 ...,approved,False,Side,...,NaN,NaN,NaN,False,NaN,CVT,yes,AT503689A1_ua1,True,0
1,Batch_02,AT503689A1,Image: AT503689A1_fig_01_crop_1_Fu.png,AT503689A1_fig_01_crop_1_Fu,1,AT503689A1_fig_01_crop_1_Fu.png,/mnt/storage_11tb/Drive_files_to_syncronize/3 ...,approved,True,Back,...,NaN,NaN,NaN,False,NaN,CVT,yes,AT503689A1_ua1,True,1
2,Batch_02,AT503689A1,Image: AT503689A1_fig_02_crop_0_F4.png,4,<NA>,AT503689A1_fig_02_crop_0_F4.png,NaN,disapproved,False,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,2
3,Batch_02,AT503689A1,Image: AT503689A1_fig_02_crop_1_Fu.png,AT503689A1_fig_02_crop_1_Fu,<NA>,AT503689A1_fig_02_crop_1_Fu.png,NaN,disapproved,False,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,3
4,Batch_02,AU2020100605A4,Image: AU2020100605A4_fig_02_crop_0_F1.png,1,<NA>,AU2020100605A4_fig_02_crop_0_F1.png,NaN,disapproved,False,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,0


## 2. Fixed gates → funnel
Each figure keeps the **first** gate that removed it (`excluded_by`).

In [3]:
cand, funnel = fs.apply_gates(cand, cfg)
funnel

,step,removed,remaining,aircraft_remaining
0,figures on file,0,5855,NaN
1,figure not approved,3940,1915,841.0
2,image file missing,0,1915,841.0
3,patent not approved,2,1913,840.0
4,no aircraft row,0,1913,840.0
5,aircraft not approved,0,1913,840.0
6,domain gate: UAVSimilar,184,1729,754.0
7,domain gate: ElectricSimilar,66,1663,719.0
8,domain gate: STOLSimilar,3,1660,718.0
9,"duplicate patent (D1/D2, points at its original)",74,1586,677.0


In [4]:
# the figures left after the gates, by their labels
elig = cand[cand['eligible']]
for col in ['acState', 'per', 'acSty', 'qualityFlag', 'is_main']:
    display(elig[col].value_counts(dropna=False).rename(col).to_frame().T)

acState,Invariant,Hover,Cruise,Transition,Other,NaN
acState,604,442,422,59,27,13


per,Front-Isometric,Top,Side,Front,Rear-Isometric,Back,Bottom/Down
per,760,314,261,118,78,26,10


acSty,Line Drawing,Render,Draft
acSty,1493,67,7


qualityFlag,clean,generic,poor_quality
qualityFlag,1557,6,4


is_main,False,True
is_main,892,675


## 3. Figure sets (one per strategy)
Sets hold at most one figure per aircraft, except `all`. If an aircraft has several candidate figures, the notebook picks in this order: the main image, then `tie_break_perspective`, then quality `clean`, then figure order.

In [5]:
sets = fs.build_sets(cand, cfg)
pd.DataFrame({k: {'figures': len(v), 'aircraft': v['aircraft_uid'].nunique()} for k, v in sets.items()}).T

[selection] state_by_type: table is empty — skipped. REMINDER: decide which flight state best shows each topType (selection.strategies.state_by_type.table).


,figures,aircraft
main,677,677
all,1567,677
hover,306,306
cruise,291,291
invariant,315,315
per_front_iso,477,477
per_side,195,195
per_top,266,266
line_only,655,655


## 4. Coverage and split
Aircraft covered by each set, broken down by topType. Strategy comparisons should use the aircraft shared by the compared sets. The `split` column divides the aircraft so the best strategy is **chosen** on `select` and **reported** on `report`.

In [6]:
aircraft = fs.aircraft_table(master, cand, cfg)
cov = fs.coverage(aircraft, sets)
display(fs.coverage_by_type(cov, list(sets)))
common = (cov[list(sets)] > 0).all(axis=1)
print('eligible aircraft:', len(aircraft), '| in every set:', int(common.sum()))
print(aircraft['split'].value_counts().to_dict())

,aircraft,main,all,hover,cruise,invariant,per_front_iso,per_side,per_top,line_only
topType,,,,,,,,,,
SLC,185,185,185,9,7,175,109,40,90,173
TR,160,160,160,134,120,4,129,56,50,158
CVT,111,111,111,91,88,2,86,25,40,109
TW,60,60,60,52,50,3,49,19,21,58
MR,50,50,50,4,7,43,30,21,23,49
TB,28,28,28,6,7,22,22,14,9,28
PTC,26,26,26,2,0,24,13,2,15,25
HB,20,20,20,0,0,20,17,6,6,19
RC,11,11,11,0,0,10,6,5,2,11


eligible aircraft: 677 | in every set: 0
{'select': 340, 'report': 337}


## Save

In [7]:
fs.save(cand, funnel, aircraft, sets, cov, cfg)

[selection] wrote 9 sets (1567 unique figures) to /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_LABELLED/2_embedding_extraction/selection


PosixPath('/mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_LABELLED/2_embedding_extraction/selection')